In [1]:
import numpy as np
import rasterio as rio
from rasterio.mask import mask
import pandas as pd
import geopandas as gpd

In [2]:
def read_check_vector(tif_path, samples_path):
    """
    Read a vector samples file and check whether its CRS matches the
    raster's CRS. If they don't match, reproject the samples to the
    raster's CRS and emit a warning.

    Parameters
    ----------
    tif_path : str or Path
        Path to the raster (.tif) file.
    samples_path : str or Path
        Path to the vector samples file (e.g., shapefile, geojson).

    Returns
    -------
    geopandas.GeoDataFrame
        The samples GeoDataFrame, reprojected to match the raster's CRS
        if needed.

    Raises
    ------
    ValueError
        If either the raster or the samples file has no CRS defined.
    """
    gdf_samples = gpd.read_file(samples_path)

    with rio.open(tif_path) as src:
        raster_crs = src.crs

    # Guard against missing CRS on either side
    if raster_crs is None:
        raise ValueError(f"Raster '{tif_path}' has no CRS defined.")
    if gdf_samples.crs is None:
        raise ValueError(f"Samples file '{samples_path}' has no CRS defined.")

    if gdf_samples.crs != raster_crs:
        warnings.warn(
            f"CRS mismatch: samples CRS ({gdf_samples.crs}) does not match "
            f"raster CRS ({raster_crs}). Reprojecting samples to match raster.",
            UserWarning,
            stacklevel=2,
        )
        gdf_samples = gdf_samples.to_crs(raster_crs)

    return gdf_samples


def clip_tif(tif_path, samples_path, samples_col, back_val=0):
    """
    Clip a raster with each polygon in a vector samples file and extract
    the pixel values within each polygon, excluding background values.

    For each polygon in the samples file, the raster is clipped to that
    polygon's extent, the specified background value is masked out (set
    to NaN and removed), and the remaining valid pixel values are stored
    in a dictionary keyed by the polygon's identifier.

    Parameters
    ----------
    tif_path : str or Path
        Path to the raster (.tif) file to clip.
    samples_path : str or Path
        Path to the vector samples file (e.g., shapefile, geojson)
        containing the polygons to clip with.
    samples_col : str
        Name of the column in the samples file to use as the identifier
        (dictionary key) for each polygon's extracted values.
    back_val : int or float, optional
        Background/nodata value to exclude from the extracted pixel
        values (default is 0).

    Returns
    -------
    dict
        Dictionary mapping each polygon's identifier (from `samples_col`)
        to a 1D numpy array of valid (non-background, non-NaN) pixel
        values extracted from that polygon.
    """
    with rio.open(tif_path) as src:
        gdf_samples = read_check_vector(tif_path, samples_path)

        dict_samp_val = {}
        for idx, row in gdf_samples.iterrows():
            geom = [row.geometry]  # mask() expects a list of geometries
            #geom = row.geometry  # mask() expects a list of geometries
            out_image, out_transform = mask(src, geom, crop=True)

            x = out_image
            x[0][x[0] == back_val] = np.nan
            x = x[~np.isnan(x)]

            dict_samp_val[row[samples_col]] = x

    return dict_samp_val

### Samples SAOCOM soil humidity

In [3]:
TIF_PATH = 'SAOCOM/Datos SAOCOM/GI_20210703212622_SAOCOM_SAR_SSMH/GI_20210703212622_SAOCOM_SAR_SSMH.tif'
SAMPLES_PATH = 'SAOCOM/Muestras suelo desnudo SAOCOM/parcelas_SueloDesnudo_SAOCOM_TP6.shp'
SAMP_COL = 'nombre'

In [12]:
df_ssmm = clip_tif(TIF_PATH, SAMPLES_PATH, SAMP_COL, back_val=0)

In [17]:
list_stats = []

for samp in df_ssmm.keys():
    list_stats.append([samp, np.mean(df_ssmm[samp]), np.std(df_ssmm[samp])])

In [18]:
df_ssmm_stats = pd.DataFrame(list_stats, columns =['nombre', 'mean', 'std'])

In [21]:
df_ssmm_stats.sample(5)

,nombre,mean,std
19,parcela_39,18.328012,2.645899
10,parcela_29,31.595959,3.422847
8,parcela_27,27.153166,2.564297
4,parcela_23,29.058798,3.548651
13,parcela_32,37.018311,4.656993


In [22]:
df_ssmm_stats.to_excel('joanala+kpa.xlsx', index=False)

### Samples angles statistics

In [4]:
TIF_PATH = 'SAOCOM/Datos SAOCOM/S1A_OPER_SAR_EOSSP__CORE_L1C_ARG-1_OLVF_20210703T230029_incidence_angle_cg.tif'

In [7]:
df_incang = clip_tif(TIF_PATH, SAMPLES_PATH, SAMP_COL, back_val=0)

In [9]:
list_stats = []

for samp in df_incang.keys():
    list_stats.append([samp, np.mean(df_incang[samp])])

In [10]:
df_incang_stats = pd.DataFrame(list_stats, columns =['nombre', 'mean'])

In [11]:
df_incang_stats.to_excel('incang_stats.xlsx', index=False)